In [ ]:
import org.apache.spark.sql.{DataFrame, SparkSession}
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

object DataExploration {

  // Configuration case class to hold settings
  case class ExplorationConfig(
      manifestFilePath: String,
      dataProductName: String,
      datasetName: String,
      datasetVersion: String
  )

  /**
   * Retrieves the full path to the IndexLatest.csv file based on the manifest configuration.
   */
  def getIndexFilePath(spark: SparkSession, config: ExplorationConfig): String = {
    import spark.implicits._

    val dfManifest = spark.read.option("multiline", "true").json(config.manifestFilePath)

    val datasetManifestEntryDf =
      dfManifest
        .select(explode($"data_products").as("dp"))
        .select(
          $"dp.name".as("dataProductName"),
          coalesce($"dp.enabled", lit(true)).as("isDataProductEnabled"),
          $"dp.output_configuration.storage_account".as("storage_account"),
          $"dp.output_configuration.container".as("container"),
          $"dp.output_configuration.base_path".as("base_path"),
          explode($"dp.tables").as("tables")
        )
        .filter(
          $"isDataProductEnabled" &&
          $"dataProductName" === config.dataProductName &&
          $"tables.name" === config.datasetName &&
          $"tables.version".cast("string") === config.datasetVersion
        )

    if (datasetManifestEntryDf.head(1).isEmpty) {
      mssparkutils.notebook.exit(
        s"Manifest file is empty/unreadable: ${config.manifestFilePath} or ${config.dataProductName} is disabled for copy."
      )
    }

    datasetManifestEntryDf
      .withColumn(
        "fullPath",
        concat(
          lit("abfss://"),
          $"container",
          lit("@"),
          $"storage_account",
          lit(".dfs.<storageEnvironmentSuffix>"),
          $"base_path",
          lit(config.datasetName),
          coalesce($"tables.sub_path", lit("")),
          lit("/index-latest.csv")
        )
      )
      .select($"fullPath")
      .as[String]
      .head()
  }

  /**
   * Reads the index file and returns the list of paths for the latest snapshot.
   */
  def getSnapshotPaths(spark: SparkSession, indexFilePath: String): Array[String] = {
    import spark.implicits._
    
    val window = Window.partitionBy("Partition").orderBy(col("RunId").desc)
    println(indexFilePath)

    spark.read
      .option("header", "true")
      .csv(indexFilePath)
      .filter(col("IngestionMode") === "Snapshot")
      .withColumn("rnk", rank().over(window))
      .filter(col("rnk") === 1)
      .select("Path").distinct.as[String].collect()
  }

  /**
   * Loads the data from the provided paths into a DataFrame.
   */
  def loadData(spark: SparkSession, paths: Array[String]): DataFrame = {
    if (paths.isEmpty) {
      spark.emptyDataFrame
    } else {
      spark.read.parquet(paths: _*)
    }
  }

  /**
   * Helper method to execute the flow with the provided configuration.
   */
  def run(spark: SparkSession, config: ExplorationConfig): DataFrame = {
    val indexFilePath = getIndexFilePath(spark, config)
    val paths = getSnapshotPaths(spark, indexFilePath)
    loadData(spark, paths)
  }
}

In [ ]:
// // Configure below parameters based on your data requirements
// val config = DataExploration.ExplorationConfig(
//   manifestFilePath = "abfss://config@<storageAccountName>.dfs.<storageEnvironmentSuffix>/manifest/v1.0/subscription-catalog.json",
//   dataProductName = "Products",
//   datasetName = "CDM_Products",
//   datasetVersion = "3.0"
// )
// val df = DataExploration.run(spark, config)